# Autoregressive Word-Level Text Generation on Tiny Shakespeare
**Foundations of Generative AI (ICT4403) — Assignment 2**

Four recurrent architectures — **Simple RNN, Bidirectional RNN, LSTM, GRU** — are trained under
identical conditions to predict the next word, and then used to generate text one word at a time
(autoregressively) from the same seed sequences at three sampling temperatures.

Pipeline: dataset → preprocessing → (input window, target word) pairs → train 4 models →
quantitative evaluation → qualitative + temperature analysis → conclusion.

In [1]:
import os, re, time, json, random, collections, urllib.request
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FAST = os.environ.get("FAST", "0") == "1"      # FAST=1 -> tiny smoke-test run

CFG = dict(
    seed=42,
    seq_len=10,          # context window: 10 previous words -> predict the 11th
    min_count=3,         # words seen fewer than 3 times in the training split -> <unk>
    emb_dim=128,
    hidden=256,
    dropout=0.4,
    batch_size=256,
    lr=1e-3,             # halved by ReduceLROnPlateau when validation loss stalls
    max_epochs=2 if FAST else 50,
    patience=5,          # early stopping on validation loss
    lr_patience=1,       # epochs without improvement before the learning rate is halved
    lr_factor=0.5,
    clip=1.0,
    train_frac=0.80, val_frac=0.10,   # remaining 10% = test
    gen_words=60,
    temperatures=[0.5, 1.0, 1.5],
)
torch.set_num_threads(os.cpu_count() or 2)
DEVICE = "cpu"
os.makedirs("results", exist_ok=True); os.makedirs("data", exist_ok=True)
print(CFG)

{'seed': 42, 'seq_len': 10, 'min_count': 3, 'emb_dim': 128, 'hidden': 256, 'dropout': 0.4, 'batch_size': 256, 'lr': 0.001, 'max_epochs': 50, 'patience': 5, 'lr_patience': 1, 'lr_factor': 0.5, 'clip': 1.0, 'train_frac': 0.8, 'val_frac': 0.1, 'gen_words': 60, 'temperatures': [0.5, 1.0, 1.5]}


## 1. Dataset
**Tiny Shakespeare** (≈1.1 MB of plain text; concatenated plays of William Shakespeare), the corpus
popularised by Andrej Karpathy's *char-rnn* project.
Source: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

Why this dataset: small enough to train four recurrent models on a CPU, yet stylistically rich
(archaic vocabulary, verse line-breaks, speaker labels), which makes coherence, diversity and
repetition easy to judge by eye.

In [2]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
PATH = "data/tinyshakespeare.txt"
if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)
raw = open(PATH, encoding="utf-8").read()
print(f"characters: {len(raw):,} | lines: {raw.count(chr(10)):,}")
print(raw[:300])

characters: 1,115,394 | lines: 40,000
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## 2. Preprocessing
1. Lower-case the text.
2. Tokenise into **words** (letters and inner apostrophes), the punctuation marks `. , ! ? ; :`
   as separate tokens, and line breaks as a `<nl>` token (verse structure is part of the style).
3. Chronological **80 / 10 / 10 split** into train / validation / test (no shuffling before the split,
   so the test text is genuinely unseen).
4. Build the vocabulary **from the training split only**; words seen fewer than 3 times become `<unk>`.
5. Build `(input window, target)` pairs with a sliding window of 10 words (stride 1):
   `x = w[t-10 .. t-1]`, `y = w[t]`.

In [3]:
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)*|[.,!?;:]|\n")

def tokenize(text):
    return ["<nl>" if t == "\n" else t for t in TOKEN_RE.findall(text.lower())]

tokens = tokenize(raw)
n = len(tokens)
n_tr, n_va = int(n * CFG["train_frac"]), int(n * CFG["val_frac"])
tok_train, tok_val, tok_test = tokens[:n_tr], tokens[n_tr:n_tr + n_va], tokens[n_tr + n_va:]

counts = collections.Counter(tok_train)
itos = ["<pad>", "<unk>"] + sorted(w for w, c in counts.items() if c >= CFG["min_count"])
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
UNK = stoi["<unk>"]

def encode(toks):
    return np.array([stoi.get(t, UNK) for t in toks], dtype=np.int64)

def make_pairs(ids, L):
    # sliding window with stride 1 -> X: (N, L), y: (N,)
    idx = np.arange(L)[None, :] + np.arange(len(ids) - L)[:, None]
    return torch.from_numpy(ids[idx]), torch.from_numpy(ids[L:])

ids_train, ids_val, ids_test = encode(tok_train), encode(tok_val), encode(tok_test)
Xtr, ytr = make_pairs(ids_train, CFG["seq_len"])
Xva, yva = make_pairs(ids_val, CFG["seq_len"])
Xte, yte = make_pairs(ids_test, CFG["seq_len"])

n_unique_all = len(set(tokens))
unk_rate_train = float((ids_train == UNK).mean())
stats = dict(
    characters=len(raw), total_tokens=n, unique_tokens_all=n_unique_all,
    vocab_size=V, unk_rate_train=round(unk_rate_train, 4),
    train_tokens=len(tok_train), val_tokens=len(tok_val), test_tokens=len(tok_test),
    train_pairs=len(Xtr), val_pairs=len(Xva), test_pairs=len(Xte),
    most_common=counts.most_common(12),
)
json.dump(stats, open("results/dataset_stats.json", "w"), indent=2)
for k, v in stats.items(): print(f"{k:>18}: {v}")
print("\nExample pair:")
print("  x ->", " ".join(itos[i] for i in Xtr[1000]))
print("  y ->", itos[int(ytr[1000])])

        characters: 1115394
      total_tokens: 290145
 unique_tokens_all: 12380
        vocab_size: 4296
    unk_rate_train: 0.036
      train_tokens: 232116
        val_tokens: 29014
       test_tokens: 29015
       train_pairs: 232106
         val_pairs: 29004
        test_pairs: 29005
       most_common: [('<nl>', 31637), (',', 15901), (':', 7997), ('.', 6123), ('the', 5195), ('and', 4518), ('to', 3947), ('i', 3608), ('of', 3194), (';', 2877), ('my', 2554), ('you', 2285)]

Example pair:
  x -> , i'll hear it , sir : yet you must
  y -> not


## 3. Models
All four models share the same skeleton so the comparison is fair:
`Embedding(128) → recurrent layer(256 units) → Dropout(0.4) → Linear(→ vocab)`.
Only the recurrent cell changes.

| Model | Cell | Notes |
|---|---|---|
| **SimpleRNN** | `tanh` Elman RNN | no gating, vanishing-gradient prone |
| **BiRNN** | `tanh` RNN, forward + backward | reads the 10-word *input window* in both directions; the two final hidden states are concatenated (512-d). This is legitimate for generation because the window contains only past words — no future token leaks |
| **LSTM** | LSTM | input/forget/output gates + cell state |
| **GRU** | GRU | reset/update gates, fewer parameters than LSTM |

In [4]:
class NextWordModel(nn.Module):
    def __init__(self, kind, V, emb, hid, drop):
        super().__init__()
        self.kind = kind
        self.emb = nn.Embedding(V, emb, padding_idx=0)
        bi = kind == "BiRNN"
        cell = {"SimpleRNN": nn.RNN, "BiRNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]
        kw = dict(nonlinearity="tanh") if kind in ("SimpleRNN", "BiRNN") else {}
        self.rnn = cell(emb, hid, batch_first=True, bidirectional=bi, **kw)
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(hid * (2 if bi else 1), V)

    def forward(self, x):
        _, h = self.rnn(self.emb(x))
        if self.kind == "LSTM": h = h[0]          # (h_n, c_n) -> h_n
        h = torch.cat([h[-2], h[-1]], dim=1) if self.kind == "BiRNN" else h[-1]
        return self.out(self.drop(h))

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

for k in ["SimpleRNN", "BiRNN", "LSTM", "GRU"]:
    print(f"{k:>10}: {n_params(NextWordModel(k, V, CFG['emb_dim'], CFG['hidden'], CFG['dropout'])):,} parameters")

 SimpleRNN: 1,752,776 parameters


     BiRNN: 2,951,368 parameters
      LSTM: 2,049,224 parameters
       GRU: 1,950,408 parameters


## 4. Training
Identical conditions for every model: same seed, data order, Adam (initial lr 1e-3, halved by
`ReduceLROnPlateau` after 1 epoch without validation improvement), batch 256, gradient clipping at 1.0,
dropout 0.4, up to 50 epochs with early stopping (patience 5) on validation loss; the checkpoint with
the best validation loss is restored. The epoch cap is only a ceiling: training length is decided by early stopping.
Training time is wall-clock time spent in training epochs on the same 2-core CPU.

In [5]:
@torch.no_grad()
def evaluate(model, X, y, bs=1024):
    model.eval()
    tot_loss, c1, c5 = 0.0, 0, 0
    for i in range(0, len(X), bs):
        logits = model(X[i:i + bs]); yb = y[i:i + bs]
        tot_loss += F.cross_entropy(logits, yb, reduction="sum").item()
        top5 = logits.topk(5, dim=1).indices
        c1 += (top5[:, 0] == yb).sum().item()
        c5 += (top5 == yb[:, None]).any(dim=1).sum().item()
    loss = tot_loss / len(X)
    return dict(loss=loss, ppl=float(np.exp(loss)), top1=c1 / len(X), top5=c5 / len(X))

def train_model(kind):
    torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"]); random.seed(CFG["seed"])
    model = NextWordModel(kind, V, CFG["emb_dim"], CFG["hidden"], CFG["dropout"])
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=CFG["lr_factor"], patience=CFG["lr_patience"])
    g = torch.Generator().manual_seed(CFG["seed"])
    history, best, best_state, bad = [], float("inf"), None, 0
    train_time = 0.0
    Xt, yt = (Xtr[:20000], ytr[:20000]) if FAST else (Xtr, ytr)
    for ep in range(1, CFG["max_epochs"] + 1):
        model.train(); t0 = time.time(); run, seen = 0.0, 0
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), CFG["batch_size"]):
            b = perm[i:i + CFG["batch_size"]]
            loss = F.cross_entropy(model(Xt[b]), yt[b])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"]); opt.step()
            run += loss.item() * len(b); seen += len(b)
        dt = time.time() - t0; train_time += dt
        va = evaluate(model, Xva, yva)
        sched.step(va["loss"])
        history.append(dict(model=kind, epoch=ep, train_loss=run / seen, val_loss=va["loss"],
                            val_ppl=va["ppl"], val_top1=va["top1"], val_top5=va["top5"], epoch_sec=dt,
                            lr=opt.param_groups[0]["lr"]))
        print(f"[{kind:>9}] ep {ep:2d} | train {run/seen:.3f} | val {va['loss']:.3f} "
              f"(ppl {va['ppl']:.1f}, top1 {va['top1']:.3f}) | {dt:.0f}s")
        if va["loss"] < best - 1e-4:
            best, bad = va["loss"], 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_ep = ep
        else:
            bad += 1
            if bad >= CFG["patience"]: break
    model.load_state_dict(best_state)
    return model, history, dict(train_time_sec=train_time, epochs_run=len(history), best_epoch=best_ep)

KINDS = ["SimpleRNN", "BiRNN", "LSTM", "GRU"]
models, all_hist, summary = {}, [], []
for kind in KINDS:
    m, hist, info = train_model(kind)
    models[kind] = m; all_hist += hist
    te, va = evaluate(m, Xte, yte), evaluate(m, Xva, yva)
    summary.append(dict(model=kind, params=n_params(m), **info,
                        sec_per_epoch=info["train_time_sec"] / info["epochs_run"],
                        val_loss=va["loss"], test_loss=te["loss"], test_ppl=te["ppl"],
                        test_top1=te["top1"], test_top5=te["top5"]))
    torch.save(m.state_dict(), f"models/{kind}.pt")

hist_df, sum_df = pd.DataFrame(all_hist), pd.DataFrame(summary)
hist_df.to_csv("results/training_history.csv", index=False)
sum_df.to_csv("results/model_comparison.csv", index=False)
print(sum_df.round(4).to_string(index=False))

[SimpleRNN] ep  1 | train 4.902 | val 4.337 (ppl 76.5, top1 0.252) | 21s


[SimpleRNN] ep  2 | train 4.435 | val 4.233 (ppl 68.9, top1 0.259) | 19s


[SimpleRNN] ep  3 | train 4.264 | val 4.212 (ppl 67.5, top1 0.265) | 19s


[SimpleRNN] ep  4 | train 4.143 | val 4.187 (ppl 65.8, top1 0.266) | 19s


[SimpleRNN] ep  5 | train 4.044 | val 4.181 (ppl 65.5, top1 0.267) | 19s


...

[      GRU] ep  6 | train 3.725 | val 4.095 (ppl 60.1, top1 0.281) | 36s


[      GRU] ep  7 | train 3.534 | val 4.090 (ppl 59.7, top1 0.282) | 36s


[      GRU] ep  8 | train 3.458 | val 4.110 (ppl 61.0, top1 0.279) | 35s


[      GRU] ep  9 | train 3.355 | val 4.117 (ppl 61.4, top1 0.279) | 35s


    model  params  train_time_sec  epochs_run  best_epoch  sec_per_epoch  val_loss  test_loss  test_ppl  test_top1  test_top5
SimpleRNN 1752776        206.0343          11           6        18.7304    4.1801     4.1386   62.7172     0.2897     0.4740
    BiRNN 2951368        245.3316           7           2        35.0474    4.2517     4.1457   63.1648     0.2824     0.4642
     LSTM 2049224        499.5979          12           7        41.6332    4.0726     4.0497   57.3782     0.3031     0.4900
      GRU 1950408        318.8308           9           4        35.4256    4.0755     4.0180   55.5878     0.2966     0.4919


## 5. Quantitative comparison
Test-set loss (cross-entropy, nats), **perplexity** = exp(loss), **top-1 / top-5 next-word accuracy**,
parameter count and training time. Perplexity here is computed over the reduced vocabulary
(including `<unk>`), so it is comparable *between these models* but not with papers using other
vocabularies or tokenisations.

In [6]:
COL = {"SimpleRNN": "#8a8d91", "BiRNN": "#e07b39", "LSTM": "#2a6fbb", "GRU": "#2f9e6b"}

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for k in KINDS:
    h = hist_df[hist_df.model == k]
    ax[0].plot(h.epoch, h.train_loss, color=COL[k], ls="--", marker="o", ms=3)
    ax[0].plot(h.epoch, h.val_loss, color=COL[k], marker="o", ms=3, label=k)
    ax[1].plot(h.epoch, h.val_ppl, color=COL[k], marker="o", ms=3, label=k)
ax[0].set(title="Loss per epoch (dashed = train, solid = validation)", xlabel="epoch", ylabel="cross-entropy")
ax[1].set(title="Validation perplexity", xlabel="epoch", ylabel="perplexity")
ax[0].legend(); ax[1].legend()
w = 0.38; xs = np.arange(len(KINDS))
ax[2].bar(xs - w/2, sum_df.test_top1 * 100, w, color="#2a6fbb", label="top-1")
ax[2].bar(xs + w/2, sum_df.test_top5 * 100, w, color="#9cc3ea", label="top-5")
ax[2].set_xticks(xs); ax[2].set_xticklabels(KINDS); ax[2].set(title="Test next-word accuracy (%)"); ax[2].legend()
for a in ax: a.grid(alpha=.25)
plt.tight_layout(); plt.savefig("results/training_curves_and_accuracy.png", dpi=150); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(KINDS, sum_df.train_time_sec, color=[COL[k] for k in KINDS]); ax[0].set_title("Total training time (s)")
ax[1].bar(KINDS, sum_df.test_ppl, color=[COL[k] for k in KINDS]); ax[1].set_title("Test perplexity (lower = better)")
for a in ax: a.grid(alpha=.25, axis="y")
plt.tight_layout(); plt.savefig("results/time_and_perplexity.png", dpi=150); plt.close()
sum_df.round(4)

,model,params,train_time_sec,epochs_run,best_epoch,sec_per_epoch,val_loss,test_loss,test_ppl,test_top1,test_top5
0,SimpleRNN,1752776,206.0343,11,6,18.7304,4.1801,4.1386,62.7172,0.2897,0.4740
1,BiRNN,2951368,245.3316,7,2,35.0474,4.2517,4.1457,63.1648,0.2824,0.4642
2,LSTM,2049224,499.5979,12,7,41.6332,4.0726,4.0497,57.3782,0.3031,0.4900
3,GRU,1950408,318.8308,9,4,35.4256,4.0755,4.0180,55.5878,0.2966,0.4919


## 6. Autoregressive generation and the effect of temperature
Generation loop: feed the last 10 words → get logits → divide by temperature `T` → softmax →
**sample** one word → append → repeat for 60 words.
* `T < 1` sharpens the distribution (safe, repetitive); `T = 1` samples from the model's own
  distribution; `T > 1` flattens it (diverse, error-prone).
* The `<unk>` and `<pad>` tokens are masked out so they are never generated.
* Every model gets the **same three seeds**, the same three temperatures (0.5, 1.0, 1.5) and the same
  random generator seed for each (seed, temperature) pair.

In [7]:
SEEDS = [
    "first citizen : we are accounted poor citizens , the patricians good .",
    "my lord , the king is dead and all the people mourn for him .",
    "o romeo , romeo ! wherefore art thou so cruel to me ,",
]

@torch.no_grad()
def generate(model, seed_text, T, n_words, rng_seed):
    model.eval()
    g = torch.Generator().manual_seed(rng_seed)
    toks = tokenize(seed_text)
    ids = [stoi.get(t, UNK) for t in toks]
    assert len(ids) >= CFG["seq_len"], "seed shorter than context window"
    out = []
    for _ in range(n_words):
        x = torch.tensor([ids[-CFG["seq_len"]:]])
        logits = model(x)[0] / T
        logits[[0, UNK]] = -float("inf")
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1, generator=g).item()
        ids.append(nxt); out.append(itos[nxt])
    return out

def detok(words):
    s = " ".join(words).replace(" <nl>", "\n").replace("<nl> ", "\n").replace("<nl>", "\n")
    s = re.sub(r" ([.,!?;:])", r"\1", s)
    return s

train_bigrams = set(zip(tok_train, tok_train[1:]))

def gen_metrics(words):
    w = [x for x in words if x != "<nl>"]
    bi = list(zip(w, w[1:])); tri = list(zip(w, w[1:], w[2:]))
    d1 = len(set(w)) / len(w)
    d2 = len(set(bi)) / max(len(bi), 1)
    rep3 = 1 - len(set(tri)) / max(len(tri), 1)        # share of repeated word-trigrams
    valid_bi = sum(b in train_bigrams for b in bi) / max(len(bi), 1)   # local-coherence proxy
    return dict(distinct1=d1, distinct2=d2, repeat_trigram=rep3, valid_bigram=valid_bi)

In [8]:
rows, samples = [], []
N_REP = 5      # extra replicate generations per cell (different RNG seeds) for stable metrics
for kind in KINDS:
    for si, seed in enumerate(SEEDS):
        for T in CFG["temperatures"]:
            for r in range(N_REP):
                words = generate(models[kind], seed, T, CFG["gen_words"], rng_seed=1000 * si + 10 * r + int(T * 10))
                rows.append(dict(model=kind, seed_id=si, T=T, rep=r, **gen_metrics(words)))
                if r == 0:
                    samples.append(dict(model=kind, seed_id=si, seed=seed, T=T, text=detok(words)))
gen_df = pd.DataFrame(rows)
gen_df.to_csv("results/generation_metrics_raw.csv", index=False)
agg = (gen_df.groupby(["model", "T"])[["distinct1", "distinct2", "repeat_trigram", "valid_bigram"]]
       .mean().reset_index())
agg["model"] = pd.Categorical(agg["model"], KINDS); agg = agg.sort_values(["model", "T"])
agg.to_csv("results/generation_metrics_by_model_temperature.csv", index=False)
print(agg.round(3).to_string(index=False))

with open("results/generated_samples.md", "w", encoding="utf-8") as f:
    f.write("# Generated text samples\n\nSame seeds and temperatures for every model "
            "(60 generated words, first replicate shown).\n")
    for si, seed in enumerate(SEEDS):
        f.write(f"\n## Seed {si+1}: \"{seed}\"\n")
        for T in CFG["temperatures"]:
            f.write(f"\n### Temperature {T}\n")
            for kind in KINDS:
                s = next(x for x in samples if x["model"] == kind and x["seed_id"] == si and x["T"] == T)
                f.write(f"\n**{kind}**\n\n```\n{s['text']}\n```\n")
json.dump(samples, open("results/generated_samples.json", "w"), indent=1)

    model   T  distinct1  distinct2  repeat_trigram  valid_bigram
SimpleRNN 0.5      0.664      0.939           0.007         0.938
SimpleRNN 1.0      0.801      0.991           0.003         0.699
SimpleRNN 1.5      0.886      1.000           0.000         0.394
    BiRNN 0.5      0.627      0.908           0.022         0.948
    BiRNN 1.0      0.808      0.995           0.000         0.655
    BiRNN 1.5      0.907      1.000           0.000         0.297
     LSTM 0.5      0.661      0.927           0.015         0.943
     LSTM 1.0      0.827      0.999           0.000         0.722
     LSTM 1.5      0.900      1.000           0.000         0.394
      GRU 0.5      0.623      0.893           0.035         0.923
      GRU 1.0      0.793      0.993           0.000         0.676
      GRU 1.5      0.884      0.999           0.000         0.356


In [9]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for k in KINDS:
    a = agg[agg.model == k]
    ax[0].plot(a["T"], a.distinct2, marker="o", color=COL[k], label=k)
    ax[1].plot(a["T"], a.repeat_trigram, marker="o", color=COL[k], label=k)
    ax[2].plot(a["T"], a.valid_bigram, marker="o", color=COL[k], label=k)
ax[0].set(title="Diversity: distinct-2 (higher = more varied)", xlabel="temperature")
ax[1].set(title="Repetition: repeated-trigram share (lower = better)", xlabel="temperature")
ax[2].set(title="Local coherence: bigrams seen in training text", xlabel="temperature")
for a in ax: a.set_xticks(CFG["temperatures"]); a.grid(alpha=.25); a.legend()
plt.tight_layout(); plt.savefig("results/temperature_analysis.png", dpi=150); plt.close()

## 7. Generated text (excerpt)
Full set of samples: `results/generated_samples.md`.

In [10]:
for s in samples:
    if s["seed_id"] == 0:
        print(f"--- {s['model']} | T={s['T']} ---\n{s['seed']}  >>  {s['text']}\n")

--- SimpleRNN | T=0.5 ---
first citizen : we are accounted poor citizens , the patricians good .  >>  

 northumberland:
 i will not be so to the king,
 and she shall be to his son,
 and i have in the time of the
 which of the world, if you be so,
 to make the consuls of my lord?

 king richard iii:
 what

--- SimpleRNN | T=1.0 ---
first citizen : we are accounted poor citizens , the patricians good .  >>  
which looks the bear he made richard profit
 comes and
 rivers:
 your head is coming, to done of his sons,
 who wear a majesty in lists two.

 florizel:

 dost shall seem died, to command that should they
 which and go on his men.

--- SimpleRNN | T=1.5 ---
first citizen : we are accounted poor citizens , the patricians good .  >>  
anger none ceremony attend well dying: but only not or madam, adieu:
 though and rice garland was deposed sceptre;
 whilst price the model as surfeit do you lartius.
 then, nay lies poison'd to black forbid
 through trudge from the language with loose pl

## 8. Results and discussion
*(The numbers below are copied from the run above; the same tables are saved in `results/`. Single run, seed 42.)*

![curves](results/training_curves_and_accuracy.png)
![time](results/time_and_perplexity.png)

### 8.1 Quantitative comparison (test split, 29,005 windows)

| Model | Params | Epochs run (best) | Train time | sec/epoch | Test loss | **Test perplexity** | Top-1 | Top-5 |
|---|---|---|---|---|---|---|---|---|
| SimpleRNN | 1.75 M | 11 (6) | 206 s | 18.7 | 4.139 | 62.7 | 29.0 % | 47.4 % |
| BiRNN | 2.95 M | 7 (2) | 245 s | 35.0 | 4.146 | 63.2 | 28.2 % | 46.4 % |
| LSTM | 2.05 M | 12 (7) | 500 s | 41.6 | 4.050 | 57.4 | **30.3 %** | 49.0 % |
| **GRU** | 1.95 M | 9 (4) | 319 s | 35.4 | **4.018** | **55.6** | 29.7 % | **49.2 %** |

* **Gated models beat the plain RNNs**: test perplexity 55.6 (GRU) and 57.4 (LSTM) vs 62.7 (SimpleRNN) and 63.2 (BiRNN), and about 1 to 3 points higher accuracy.
* **GRU vs LSTM is close.** GRU has the lower test perplexity (55.6 vs 57.4, 3 % lower) and slightly higher top-5; LSTM has the higher top-1 (30.3 % vs 29.7 %). Validation loss is essentially identical (4.073 LSTM vs 4.076 GRU at their best epochs). GRU is cheaper: 35.4 vs 41.6 s per epoch (about 15 % faster), 5 % fewer parameters, and 319 s vs 500 s of total training time.
* **BiRNN does not help.** It is no better than the SimpleRNN (test perplexity 63.2 vs 62.7, lower top-1), has about 70 % more parameters, takes about twice as long per epoch, and overfits fastest (best epoch 2; training loss 3.30 vs validation 4.27 at epoch 7). The 10-word window only contains *past* words and the next word depends mostly on the most recent ones, so reading the window backwards most likely adds little signal while adding capacity to memorise (we did not run an ablation to confirm this).
* **Training longer did not help.** With a lower learning rate (1e-3, halved on plateau), dropout 0.4 and patience 5, the models trained for 7 to 12 epochs, but the best validation loss was still reached at epoch 2 to 7. After that, training loss kept falling while validation loss rose (for example LSTM at epoch 12: training 3.28 vs validation 4.15). The corpus is small (232 k training tokens), so the models memorise it quickly; the early-stopping checkpoint is what is reported.
* Perplexity is only comparable across these four models (same vocabulary and tokenisation).

### 8.2 Qualitative comparison of generated text
Full samples: `results/generated_samples.md`. All models received the same seeds and the same sampling random numbers.

* **Coherence.** At T = 0.5 every model produces Shakespeare-shaped text: speaker labels followed by a colon, line breaks, verse-like clauses. The **LSTM** sample for the first seed is the most coherent ("northumberland: i will not believe it. / first conspirator: and to the palace, if it be so"). The **GRU** keeps the turn-taking structure but makes more grammatical slips ("nurse: o, the prince did i be so long."). **SimpleRNN and BiRNN** fall back on generic phrases ("to make his son, as this / in the time of his heart"). None of the models stays on a topic for 60 words: the 10-word window gives no long-range memory, so *contextual consistency* is weak for all of them and mostly limited to style and speaker-label format.
* **Local coherence metric** (share of generated bigrams that occur in the training text): at T = 0.5 the models are within 0.025 of each other (0.923 to 0.948); at T = 1.0 LSTM is highest (0.722) and BiRNN lowest (0.655); at T = 1.5 LSTM and SimpleRNN tie (0.394) and BiRNN is lowest (0.297).
* **Diversity.** distinct-1 rises with temperature for all models (about 0.62 to 0.66 at T = 0.5, 0.79 to 0.83 at T = 1.0, 0.88 to 0.91 at T = 1.5). distinct-2 is near 1.0 for T ≥ 1, so it separates models only at T = 0.5, where SimpleRNN is the most varied (0.939) and GRU the least (0.893). With 60-word samples distinct-2 is a weak discriminator.
* **Repetition.** Repetition is low overall and only visible at T = 0.5: repeated-trigram share is 0.7 % (SimpleRNN), 1.5 % (LSTM), 2.2 % (BiRNN) and 3.5 % (GRU). The GRU repeats most at low temperature; at T ≥ 1 repetition is about 0 for all models.
* **Same random numbers, same text.** Because every model uses the same random numbers, the samples at T = 1.0 and especially T = 1.5 start very similarly across models (for example "anger none ceremony attend well dying ..."). At high temperature the sampling noise dominates and the model matters little.

### 8.3 Effect of sampling temperature (T = 0.5, 1.0, 1.5)

![temperature](results/temperature_analysis.png)

| Model | T = 0.5 / 1.0 / 1.5 | distinct-1 | distinct-2 | repeated trigrams | valid bigrams |
|---|---|---|---|---|---|
| SimpleRNN | | 0.664 / 0.801 / 0.886 | 0.939 / 0.991 / 1.000 | 0.007 / 0.003 / 0.000 | 0.938 / 0.699 / 0.394 |
| BiRNN | | 0.627 / 0.808 / 0.907 | 0.908 / 0.995 / 1.000 | 0.022 / 0.000 / 0.000 | 0.948 / 0.655 / 0.297 |
| LSTM | | 0.661 / 0.827 / 0.900 | 0.927 / 0.999 / 1.000 | 0.015 / 0.000 / 0.000 | 0.943 / 0.722 / 0.394 |
| GRU | | 0.623 / 0.793 / 0.884 | 0.893 / 0.993 / 0.999 | 0.035 / 0.000 / 0.000 | 0.923 / 0.676 / 0.356 |

(Mean over 3 seeds × 5 replicate generations per model and temperature.)

* **T = 0.5, conservative.** Highest coherence (92 to 95 % of bigrams are real training bigrams) and the most grammatical text, but the lowest diversity and the only temperature with measurable repetition (stock phrases such as "to the king", "my lord", "king richard iii:").
* **T = 1.0, balanced.** Samples from the model's own distribution: varied and often readable phrase by phrase, with ungrammatical joins. Valid-bigram share falls to 0.66 to 0.72.
* **T = 1.5, too hot.** No repetition and maximum diversity, but only 30 to 39 % of bigrams occur in the training text and the output is word salad ("anger none ceremony attend well dying: but only not or madam, adieu"). Diversity here comes from noise, not creativity.
* **Trade-off.** Raising T trades coherence for diversity and removes repetition. Temperature has a larger effect than the choice of model: going from T = 0.5 to 1.5 drops valid bigrams by 0.55 to 0.65, while the gap between models at one temperature is at most about 0.10. Of the three temperatures tested, 0.5 gives the most readable text and 1.0 the best balance; a value between them (about 0.7) is a reasonable guess for the best compromise, but it was not tested.

## 9. Conclusion: best model
**The GRU is the most effective model for this dataset overall, with the LSTM a very close second.**

* GRU: lowest test perplexity (55.6 vs 57.4 for LSTM), best top-5 accuracy (49.2 %), about 15 % faster per epoch, 36 % less total training time (319 s vs 500 s) and slightly fewer parameters.
* LSTM: slightly higher top-1 accuracy (30.3 % vs 29.7 %), the same validation loss, and in the generated text a little more coherent at T = 1.0 and less repetitive at T = 0.5 (valid bigrams 0.722 vs 0.676; repeated trigrams 1.5 % vs 3.5 %). If the quality of individual samples matters more than cost, the LSTM is a reasonable choice.
* SimpleRNN (test perplexity 62.7) is clearly behind because it has no gating; BiRNN (63.2) gives no benefit over it and is more expensive, because a backward pass over a window of past words adds parameters without much extra information.

**Recommended setting:** GRU (or LSTM) at a temperature between 0.5 and 1.0.

**Limitations.** One training run per model (seed 42), so the GRU-LSTM gap is within run-to-run noise: an earlier run of this notebook with a 90/5/5 split and different training settings (lr 2e-3, dropout 0.3, patience 2) gave the opposite order (LSTM slightly ahead), so the two should be treated as tied. The context window is only 10 words, which limits long-range coherence for all models. The generation metrics (valid-bigram share, distinct-n, repeated-trigram share) are proxies computed on 15 short generations per cell, and the qualitative reading is by inspection of a handful of samples. Possible follow-ups: several seeds, longer context, top-k or nucleus sampling.